# بارگذاری داده‌ها

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

COL_TABLE = "نام جدول"
COL_COLUMN = "نام ستون"
COL_DTYPE = "نوع داده"
COL_ROW_COUNT = "تعداد ردیف"
COL_MISSING_COUNT = "تعداد مقدار گمشده"
COL_MISSING_PERCENT = "درصد مقدار گمشده"

candidate_data_dirs = [
    Path("../data/dirty_data"),
    Path("data/dirty_data"),
]

data_dir = next((path for path in candidate_data_dirs if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError("پوشه داده‌های خام پیدا نشد.")

csv_files = sorted(data_dir.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"هیچ فایل CSV در {data_dir} پیدا نشد.")

tables = {}
for file_path in csv_files:
    tables[file_path.stem] = pd.read_csv(file_path)

if not tables:
    raise ValueError("هیچ جدولی از فایل‌های CSV خوانده نشد.")

project_root = data_dir.parent.parent.resolve()

print(f"مسیر داده‌ها: {data_dir.resolve()}")
print(f"تعداد فایل‌های CSV: {len(csv_files)}")
for table_name, df in tables.items():
    print(f"جدول {table_name}: {df.shape}")


مسیر داده‌ها: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\data\dirty_data
تعداد فایل‌های CSV: 13
جدول Categories: (9, 4)
جدول CustomerCustomerDemo: (0, 2)
جدول CustomerDemographics: (0, 2)
جدول Customers: (93, 11)
جدول Employees: (10, 18)
جدول EmployeeTerritories: (54, 2)
جدول Order_Details: (2165, 5)
جدول Orders: (835, 14)
جدول Products: (79, 10)
جدول Region: (5, 2)
جدول Shippers: (4, 3)
جدول Suppliers: (32, 12)
جدول Territories: (59, 3)


# بررسی مقادیر گمشده

In [2]:
missing_rows = []

for table_name, df in tables.items():
    total_rows = len(df)

    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        missing_percent = (missing_count / total_rows) * 100 if total_rows > 0 else 0

        missing_rows.append({
            COL_TABLE: table_name,
            COL_COLUMN: column,
            COL_DTYPE: str(df[column].dtype),
            COL_ROW_COUNT: int(total_rows),
            COL_MISSING_COUNT: missing_count,
            COL_MISSING_PERCENT: float(missing_percent),
        })

missing_values_report = pd.DataFrame(missing_rows)
expected_columns = [COL_TABLE, COL_COLUMN, COL_DTYPE, COL_ROW_COUNT, COL_MISSING_COUNT, COL_MISSING_PERCENT]
missing_values_report = missing_values_report.loc[:, expected_columns]

assert not missing_values_report.empty
assert missing_values_report.shape[1] == 6
assert COL_MISSING_PERCENT in missing_values_report.columns

print(missing_values_report.shape)
print(missing_values_report.columns.tolist())
display(missing_values_report.head())
display(missing_values_report)


(88, 6)
['نام جدول', 'نام ستون', 'نوع داده', 'تعداد ردیف', 'تعداد مقدار گمشده', 'درصد مقدار گمشده']


,نام جدول,نام ستون,نوع داده,تعداد ردیف,تعداد مقدار گمشده,درصد مقدار گمشده
0,Categories,CategoryID,int64,9,0,0.000000
1,Categories,CategoryName,object,9,0,0.000000
2,Categories,Description,object,9,4,44.444444
3,Categories,Picture,object,9,0,0.000000
4,CustomerCustomerDemo,CustomerID,object,0,0,0.000000


,نام جدول,نام ستون,نوع داده,تعداد ردیف,تعداد مقدار گمشده,درصد مقدار گمشده
0,Categories,CategoryID,int64,9,0,0.000000
1,Categories,CategoryName,object,9,0,0.000000
2,Categories,Description,object,9,4,44.444444
3,Categories,Picture,object,9,0,0.000000
4,CustomerCustomerDemo,CustomerID,object,0,0,0.000000
...,...,...,...,...,...,...
83,Suppliers,Fax,object,32,22,68.750000
84,Suppliers,HomePage,object,32,0,0.000000
85,Territories,TerritoryID,int64,59,0,0.000000
86,Territories,TerritoryDescription,object,59,0,0.000000


# ستون‌های دارای Missing بالا

In [3]:
high_missing_columns = missing_values_report[
    missing_values_report[COL_MISSING_PERCENT] > 30
].copy()

if high_missing_columns.empty:
    print("ستونی با Missing بیش از ۳۰ درصد پیدا نشد.")

display(high_missing_columns)


,نام جدول,نام ستون,نوع داده,تعداد ردیف,تعداد مقدار گمشده,درصد مقدار گمشده
2,Categories,Description,object,9,4,44.444444
14,Customers,Region,object,93,61,65.591398
18,Customers,Fax,object,93,31,33.333333
28,Employees,Region,object,10,5,50.000000
55,Orders,ShipRegion,object,835,511,61.197605
62,Products,QuantityPerUnit,object,79,26,32.911392
79,Suppliers,Region,object,32,22,68.750000
83,Suppliers,Fax,object,32,22,68.750000


# مقایسه Mean و Median

In [4]:
numeric_missing_rows = []

for table_name, df in tables.items():
    numeric_columns = df.select_dtypes(include=[np.number]).columns

    for column in numeric_columns:
        missing_count = int(df[column].isna().sum())
        if missing_count == 0:
            continue

        total_rows = len(df)
        missing_percent = (missing_count / total_rows) * 100 if total_rows > 0 else 0
        mean_value = df[column].mean()
        median_value = df[column].median()

        numeric_missing_rows.append({
            COL_TABLE: table_name,
            COL_COLUMN: column,
            "تعداد Missing": missing_count,
            "درصد Missing": float(missing_percent),
            "Mean": mean_value,
            "Median": median_value,
            "اختلاف Mean و Median": abs(mean_value - median_value) if pd.notna(mean_value) and pd.notna(median_value) else np.nan,
        })

numeric_missing_summary = pd.DataFrame(numeric_missing_rows, columns=[
    COL_TABLE, COL_COLUMN, "تعداد Missing", "درصد Missing", "Mean", "Median", "اختلاف Mean و Median"
])

display(numeric_missing_summary)


,نام جدول,نام ستون,تعداد Missing,درصد Missing,Mean,Median,اختلاف Mean و Median
0,Employees,ReportsTo,1,10.000000,4.444444,2.0,2.444444
1,Order_Details,UnitPrice,6,0.277136,28.619444,18.4,10.219444
2,Order_Details,Quantity,15,0.692841,25.002791,20.0,5.002791
3,Order_Details,Discount,19,0.877598,0.058616,0.0,0.058616


# بررسی داده‌های تکراری

In [5]:
duplicate_rows = []

for table_name, df in tables.items():
    total_rows = len(df)
    duplicate_count = int(df.duplicated().sum())
    duplicate_percent = (duplicate_count / total_rows) * 100 if total_rows > 0 else 0

    duplicate_rows.append({
        COL_TABLE: table_name,
        COL_ROW_COUNT: int(total_rows),
        "تعداد ردیف تکراری": duplicate_count,
        "درصد ردیف تکراری": float(duplicate_percent),
    })

duplicate_rows_report = pd.DataFrame(duplicate_rows)
display(duplicate_rows_report)


,نام جدول,تعداد ردیف,تعداد ردیف تکراری,درصد ردیف تکراری
0,Categories,9,1,11.111111
1,CustomerCustomerDemo,0,0,0.000000
2,CustomerDemographics,0,0,0.000000
3,Customers,93,0,0.000000
4,Employees,10,0,0.000000
5,EmployeeTerritories,54,3,5.555556
6,Order_Details,2165,9,0.415704
7,Orders,835,0,0.000000
8,Products,79,1,1.265823
9,Region,5,0,0.000000


# بررسی ستون‌های متنی

In [6]:
text_whitespace_rows = []

for table_name, df in tables.items():
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        values = df[column].dropna().astype(str)
        total_values = len(values)
        whitespace_count = int(values.ne(values.str.strip()).sum())
        whitespace_percent = (whitespace_count / total_values) * 100 if total_values > 0 else 0

        text_whitespace_rows.append({
            COL_TABLE: table_name,
            COL_COLUMN: column,
            "تعداد مقدار دارای فاصله اضافی": whitespace_count,
            "درصد مقدار دارای فاصله اضافی": float(whitespace_percent),
        })

text_whitespace_report = pd.DataFrame(text_whitespace_rows, columns=[
    COL_TABLE, COL_COLUMN,
    "تعداد مقدار دارای فاصله اضافی",
    "درصد مقدار دارای فاصله اضافی",
])

display(text_whitespace_report)


,نام جدول,نام ستون,تعداد مقدار دارای فاصله اضافی,درصد مقدار دارای فاصله اضافی
0,Categories,CategoryName,3,33.333333
1,Categories,Description,0,0.000000
2,Categories,Picture,0,0.000000
3,CustomerCustomerDemo,CustomerID,0,0.000000
4,CustomerCustomerDemo,CustomerTypeID,0,0.000000
5,CustomerDemographics,CustomerTypeID,0,0.000000
6,CustomerDemographics,CustomerDesc,0,0.000000
7,Customers,CustomerID,0,0.000000
8,Customers,CompanyName,20,21.505376
9,Customers,ContactName,15,16.129032


# بررسی جدول Order_Details

In [7]:
def normalize_table_name(name):
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")

order_details_name = next((table_name for table_name in tables if normalize_table_name(table_name) == "orderdetails"), None)
order_details_found = order_details_name is not None

if order_details_found:
    order_details_df = tables[order_details_name]
    total_rows = len(order_details_df)
    order_quality_rows = []

    for column, check_name, condition_type in [
        ("Quantity", "Quantity <= 0", "non_positive"),
        ("UnitPrice", "UnitPrice <= 0", "non_positive"),
        ("Quantity", "Quantity Missing", "missing"),
        ("UnitPrice", "UnitPrice Missing", "missing"),
    ]:
        if column not in order_details_df.columns:
            order_quality_rows.append({"بررسی": check_name, "تعداد": np.nan, "درصد": np.nan, "پیام": f"ستون {column} پیدا نشد."})
            continue

        if condition_type == "missing":
            issue_count = int(order_details_df[column].isna().sum())
        else:
            numeric_values = pd.to_numeric(order_details_df[column], errors="coerce")
            issue_count = int((numeric_values <= 0).sum())

        issue_percent = (issue_count / total_rows) * 100 if total_rows > 0 else 0
        order_quality_rows.append({"بررسی": check_name, "تعداد": issue_count, "درصد": float(issue_percent), "پیام": f"در جدول {order_details_name} بررسی شد."})

    order_details_quality_summary = pd.DataFrame(order_quality_rows)
else:
    order_details_quality_summary = pd.DataFrame([{"پیام": "جدول Order_Details پیدا نشد."}])

display(order_details_quality_summary)


,بررسی,تعداد,درصد,پیام
0,Quantity <= 0,10,0.461894,در جدول Order_Details بررسی شد.
1,UnitPrice <= 0,6,0.277136,در جدول Order_Details بررسی شد.
2,Quantity Missing,15,0.692841,در جدول Order_Details بررسی شد.
3,UnitPrice Missing,6,0.277136,در جدول Order_Details بررسی شد.


# ذخیره گزارش‌های اولیه

In [8]:
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

report_outputs = {
    "missing_values_report.csv": missing_values_report,
    "high_missing_columns.csv": high_missing_columns,
    "numeric_missing_summary.csv": numeric_missing_summary,
    "duplicate_rows_report.csv": duplicate_rows_report,
    "text_whitespace_report.csv": text_whitespace_report,
    "order_details_quality_summary.csv": order_details_quality_summary,
}

for file_name, report_df in report_outputs.items():
    report_df.to_csv(reports_dir / file_name, index=False, encoding="utf-8-sig")
    print(f"ذخیره شد: {reports_dir / file_name}")


ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\missing_values_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\high_missing_columns.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\numeric_missing_summary.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\duplicate_rows_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\text_whitespace_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\order_details_quality_summary.csv


# بررسی پایانی اولیه

In [9]:
total_checked_columns = int(sum(len(df.columns) for df in tables.values()))
missing_column_count = int((missing_values_report[COL_MISSING_COUNT] > 0).sum())
high_missing_count = int(len(high_missing_columns))
duplicate_table_count = int((duplicate_rows_report["تعداد ردیف تکراری"] > 0).sum())
order_details_status = "پیدا شد" if order_details_found else "پیدا نشد"

print(f"تعداد جدول‌های خوانده‌شده: {len(tables)}")
print(f"تعداد کل ستون‌های بررسی‌شده: {total_checked_columns}")
print(f"تعداد ستون‌های دارای Missing: {missing_column_count}")
print(f"تعداد ستون‌های دارای Missing بالای ۳۰ درصد: {high_missing_count}")
print(f"تعداد جدول‌های دارای Duplicate: {duplicate_table_count}")
print(f"وضعیت جدول Order_Details: {order_details_status}")
print("داده‌های اصلی تغییر داده نشده‌اند.")


تعداد جدول‌های خوانده‌شده: 13
تعداد کل ستون‌های بررسی‌شده: 88
تعداد ستون‌های دارای Missing: 23
تعداد ستون‌های دارای Missing بالای ۳۰ درصد: 8
تعداد جدول‌های دارای Duplicate: 5
وضعیت جدول Order_Details: پیدا شد
داده‌های اصلی تغییر داده نشده‌اند.


# پاکسازی داده‌ها

In [10]:
tables_clean = {
    name: df.copy()
    for name, df in tables.items()
}


# پاکسازی متن‌ها

In [11]:
text_cleaning_rows = []

for table_name, df in tables_clean.items():
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        original_values = df[column]
        non_missing_values = original_values.dropna()
        cleaned_values = non_missing_values.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
        changed_count = int((non_missing_values.astype(str) != cleaned_values).sum())
        df.loc[non_missing_values.index, column] = cleaned_values

        text_cleaning_rows.append({COL_TABLE: table_name, COL_COLUMN: column, "تعداد مقادیر اصلاح‌شده": changed_count})

text_cleaning_report = pd.DataFrame(text_cleaning_rows, columns=[COL_TABLE, COL_COLUMN, "تعداد مقادیر اصلاح‌شده"])
display(text_cleaning_report)


,نام جدول,نام ستون,تعداد مقادیر اصلاح‌شده
0,Categories,CategoryName,3
1,Categories,Description,0
2,Categories,Picture,2
3,CustomerCustomerDemo,CustomerID,0
4,CustomerCustomerDemo,CustomerTypeID,0
5,CustomerDemographics,CustomerTypeID,0
6,CustomerDemographics,CustomerDesc,0
7,Customers,CustomerID,0
8,Customers,CompanyName,40
9,Customers,ContactName,39


# حذف ردیف‌های کاملا تکراری

In [12]:
duplicate_removal_rows = []

for table_name, df in tables_clean.items():
    rows_before = int(len(df))
    tables_clean[table_name] = df.drop_duplicates().reset_index(drop=True)
    rows_after = int(len(tables_clean[table_name]))
    removed_rows = rows_before - rows_after
    duplicate_removal_rows.append({"نام جدول": table_name, "تعداد ردیف قبل": rows_before, "تعداد ردیف بعد": rows_after, "تعداد ردیف حذف‌شده": removed_rows})

duplicate_removal_report = pd.DataFrame(duplicate_removal_rows)
display(duplicate_removal_report)


,نام جدول,تعداد ردیف قبل,تعداد ردیف بعد,تعداد ردیف حذف‌شده
0,Categories,9,8,1
1,CustomerCustomerDemo,0,0,0
2,CustomerDemographics,0,0,0
3,Customers,93,93,0
4,Employees,10,10,0
5,EmployeeTerritories,54,51,3
6,Order_Details,2165,2156,9
7,Orders,835,835,0
8,Products,79,78,1
9,Region,5,4,1


# جایگزینی مقادیر گمشده

In [13]:
missing_imputation_rows = []

for table_name, df in tables_clean.items():
    total_rows = len(df)

    for column in df.columns:
        missing_before = int(df[column].isna().sum())
        if missing_before == 0:
            continue

        missing_percent_before = (missing_before / total_rows) * 100 if total_rows > 0 else 0
        dtype_before = str(df[column].dtype)

        if "discount" in str(column).lower():
            df[column] = df[column].fillna(0)
            method = "جایگزینی با صفر"
        elif pd.api.types.is_numeric_dtype(df[column]):
            mean_value = df[column].mean()
            median_value = df[column].median()
            fill_value = median_value if pd.notna(median_value) else mean_value
            df[column] = df[column].fillna(fill_value)
            method = f"جایگزینی با میانه: {median_value}"
        else:
            df[column] = df[column].fillna("Unknown")
            method = "جایگزینی با Unknown"

        missing_after = int(df[column].isna().sum())
        missing_imputation_rows.append({
            COL_TABLE: table_name,
            COL_COLUMN: column,
            COL_DTYPE: dtype_before,
            "تعداد Missing قبل": missing_before,
            "درصد Missing قبل": float(missing_percent_before),
            "روش جایگزینی": method,
            "تعداد Missing بعد": missing_after,
        })

missing_imputation_report = pd.DataFrame(missing_imputation_rows, columns=[COL_TABLE, COL_COLUMN, COL_DTYPE, "تعداد Missing قبل", "درصد Missing قبل", "روش جایگزینی", "تعداد Missing بعد"])
display(missing_imputation_report)


,نام جدول,نام ستون,نوع داده,تعداد Missing قبل,درصد Missing قبل,روش جایگزینی,تعداد Missing بعد
0,Categories,Description,object,4,50.000000,جایگزینی با Unknown,0
1,Customers,ContactTitle,object,4,4.301075,جایگزینی با Unknown,0
2,Customers,Region,object,61,65.591398,جایگزینی با Unknown,0
3,Customers,PostalCode,object,11,11.827957,جایگزینی با Unknown,0
4,Customers,Fax,object,31,33.333333,جایگزینی با Unknown,0
5,Employees,FirstName,object,1,10.000000,جایگزینی با Unknown,0
6,Employees,TitleOfCourtesy,object,2,20.000000,جایگزینی با Unknown,0
7,Employees,Region,object,5,50.000000,جایگزینی با Unknown,0
8,Employees,Photo,object,1,10.000000,جایگزینی با Unknown,0
9,Employees,Notes,object,3,30.000000,جایگزینی با Unknown,0


# اصلاح جدول Order_Details

In [14]:
order_details_cleaning_rows = []
order_details_cleaned_name = next((table_name for table_name in tables_clean if normalize_table_name(table_name) == "orderdetails"), None)

if order_details_cleaned_name is not None:
    order_details_clean = tables_clean[order_details_cleaned_name]

    if "Discount" in order_details_clean.columns:
        discount_missing_before = int(order_details_clean["Discount"].isna().sum())
        order_details_clean["Discount"] = order_details_clean["Discount"].fillna(0)
        order_details_cleaning_rows.append({"نوع مشکل": "مقدار گمشده Discount", "تعداد قبل": discount_missing_before, "اقدام انجام‌شده": "جایگزینی با صفر", "تعداد بعد": int(order_details_clean["Discount"].isna().sum())})

        discount_below_before = int((order_details_clean["Discount"] < 0).sum())
        order_details_clean.loc[order_details_clean["Discount"] < 0, "Discount"] = 0
        order_details_cleaning_rows.append({"نوع مشکل": "کمتر از صفر Discount", "تعداد قبل": discount_below_before, "اقدام انجام‌شده": "تنظیم روی صفر", "تعداد بعد": int((order_details_clean["Discount"] < 0).sum())})

        discount_above_before = int((order_details_clean["Discount"] > 1).sum())
        order_details_clean.loc[order_details_clean["Discount"] > 1, "Discount"] = 1
        order_details_cleaning_rows.append({"نوع مشکل": "بیشتر از یک Discount", "تعداد قبل": discount_above_before, "اقدام انجام‌شده": "تنظیم روی یک", "تعداد بعد": int((order_details_clean["Discount"] > 1).sum())})

    if "Quantity" in order_details_clean.columns:
        quantity_before = int((order_details_clean["Quantity"] <= 0).sum())
        order_details_clean = order_details_clean[order_details_clean["Quantity"] > 0].copy()
        order_details_cleaning_rows.append({"نوع مشکل": "Quantity کمتر یا مساوی صفر", "تعداد قبل": quantity_before, "اقدام انجام‌شده": "حذف ردیف", "تعداد بعد": int((order_details_clean["Quantity"] <= 0).sum())})

    if "UnitPrice" in order_details_clean.columns:
        unit_price_before = int((order_details_clean["UnitPrice"] <= 0).sum())
        order_details_clean = order_details_clean[order_details_clean["UnitPrice"] > 0].copy()
        order_details_cleaning_rows.append({"نوع مشکل": "UnitPrice کمتر یا مساوی صفر", "تعداد قبل": unit_price_before, "اقدام انجام‌شده": "حذف ردیف", "تعداد بعد": int((order_details_clean["UnitPrice"] <= 0).sum())})

    for column in ["Quantity", "UnitPrice"]:
        if column in order_details_clean.columns:
            missing_before = int(order_details_clean[column].isna().sum())
            median_value = order_details_clean[column].median()
            order_details_clean[column] = order_details_clean[column].fillna(median_value)
            order_details_cleaning_rows.append({"نوع مشکل": f"مقدار گمشده {column}", "تعداد قبل": missing_before, "اقدام انجام‌شده": "جایگزینی با میانه", "تعداد بعد": int(order_details_clean[column].isna().sum())})

    tables_clean[order_details_cleaned_name] = order_details_clean.reset_index(drop=True)
else:
    order_details_cleaning_rows.append({"نوع مشکل": "جدول Order_Details پیدا نشد", "تعداد قبل": 0, "اقدام انجام‌شده": "اقدامی انجام نشد", "تعداد بعد": 0})

order_details_cleaning_report = pd.DataFrame(order_details_cleaning_rows, columns=["نوع مشکل", "تعداد قبل", "اقدام انجام‌شده", "تعداد بعد"])
display(order_details_cleaning_report)


,نوع مشکل,تعداد قبل,اقدام انجام‌شده,تعداد بعد
0,مقدار گمشده Discount,0,جایگزینی با صفر,0
1,کمتر از صفر Discount,3,تنظیم روی صفر,0
2,بیشتر از یک Discount,4,تنظیم روی یک,0
3,Quantity کمتر یا مساوی صفر,10,حذف ردیف,0
4,UnitPrice کمتر یا مساوی صفر,6,حذف ردیف,0
5,مقدار گمشده Quantity,0,جایگزینی با میانه,0
6,مقدار گمشده UnitPrice,0,جایگزینی با میانه,0


# بررسی مجدد بعد از پاکسازی

In [15]:
cleaning_after_check_rows = []

for table_name, df in tables_clean.items():
    cleaning_after_check_rows.append({
        COL_TABLE: table_name,
        "تعداد Missing باقی‌مانده": int(df.isna().sum().sum()),
        "تعداد Duplicate باقی‌مانده": int(df.duplicated().sum()),
    })

cleaning_after_check_report = pd.DataFrame(cleaning_after_check_rows)

if order_details_cleaned_name is not None:
    order_details_after_df = tables_clean[order_details_cleaned_name]
    order_details_after_check = pd.DataFrame([{
        COL_TABLE: order_details_cleaned_name,
        "تعداد Missing باقی‌مانده": int(order_details_after_df.isna().sum().sum()),
        "تعداد Duplicate باقی‌مانده": int(order_details_after_df.duplicated().sum()),
        "تعداد Quantity نامعتبر": int((order_details_after_df["Quantity"] <= 0).sum()) if "Quantity" in order_details_after_df.columns else np.nan,
        "تعداد UnitPrice نامعتبر": int((order_details_after_df["UnitPrice"] <= 0).sum()) if "UnitPrice" in order_details_after_df.columns else np.nan,
        "تعداد Discount نامعتبر": int(((order_details_after_df["Discount"] < 0) | (order_details_after_df["Discount"] > 1)).sum()) if "Discount" in order_details_after_df.columns else np.nan,
    }])
else:
    order_details_after_check = pd.DataFrame([{"پیام": "جدول Order_Details پیدا نشد"}])

display(cleaning_after_check_report)
display(order_details_after_check)


,نام جدول,تعداد Missing باقی‌مانده,تعداد Duplicate باقی‌مانده
0,Categories,0,0
1,CustomerCustomerDemo,0,0
2,CustomerDemographics,0,0
3,Customers,0,0
4,Employees,0,0
5,EmployeeTerritories,0,0
6,Order_Details,0,0
7,Orders,0,0
8,Products,0,0
9,Region,0,0


,نام جدول,تعداد Missing باقی‌مانده,تعداد Duplicate باقی‌مانده,تعداد Quantity نامعتبر,تعداد UnitPrice نامعتبر,تعداد Discount نامعتبر
0,Order_Details,0,0,0,0,0


# گزارش نهایی پاکسازی

In [16]:
text_cleaning_total = int(text_cleaning_report["تعداد مقادیر اصلاح‌شده"].sum())
duplicate_removed_total = int(duplicate_removal_report["تعداد ردیف حذف‌شده"].sum())
missing_imputed_total = int(missing_imputation_report["تعداد Missing قبل"].sum()) if not missing_imputation_report.empty else 0

def action_count(pattern):
    rows = order_details_cleaning_report[order_details_cleaning_report["نوع مشکل"].str.contains(pattern, regex=False, na=False)]
    return int(rows["تعداد قبل"].sum()) if not rows.empty else 0

cleaning_actions_summary = pd.DataFrame([
    {"بخش": "پاکسازی متن", "اقدام انجام‌شده": "حذف فاصله‌های اضافی", "تعداد اصلاح‌شده": text_cleaning_total, "توضیح": "مقادیر NaN حفظ شدند"},
    {"بخش": "حذف تکراری‌ها", "اقدام انجام‌شده": "حذف ردیف‌های کاملا تکراری", "تعداد اصلاح‌شده": duplicate_removed_total, "توضیح": "فقط روی tables_clean"},
    {"بخش": "جایگزینی Missing", "اقدام انجام‌شده": "جایگزینی با صفر، میانه یا Unknown", "تعداد اصلاح‌شده": missing_imputed_total, "توضیح": "هیچ ستونی حذف نشد"},
    {"بخش": "اصلاح Discount", "اقدام انجام‌شده": "تنظیم مقادیر نامعتبر", "تعداد اصلاح‌شده": action_count("Discount"), "توضیح": "بازه صفر تا یک حفظ شد"},
    {"بخش": "اصلاح Quantity", "اقدام انجام‌شده": "حذف ردیف‌های نامعتبر", "تعداد اصلاح‌شده": action_count("Quantity"), "توضیح": "فقط در Order_Details"},
    {"بخش": "اصلاح UnitPrice", "اقدام انجام‌شده": "حذف ردیف‌های نامعتبر", "تعداد اصلاح‌شده": action_count("UnitPrice"), "توضیح": "فقط در Order_Details"},
])

display(cleaning_actions_summary)


,بخش,اقدام انجام‌شده,تعداد اصلاح‌شده,توضیح
0,پاکسازی متن,حذف فاصله‌های اضافی,1302,مقادیر NaN حفظ شدند
1,حذف تکراری‌ها,حذف ردیف‌های کاملا تکراری,19,فقط روی tables_clean
2,جایگزینی Missing,جایگزینی با صفر، میانه یا Unknown,817,هیچ ستونی حذف نشد
3,اصلاح Discount,تنظیم مقادیر نامعتبر,7,بازه صفر تا یک حفظ شد
4,اصلاح Quantity,حذف ردیف‌های نامعتبر,10,فقط در Order_Details
5,اصلاح UnitPrice,حذف ردیف‌های نامعتبر,6,فقط در Order_Details


# ذخیره گزارش‌های پاکسازی

In [17]:
cleaning_reports_dir = project_root / "reports"
cleaning_reports_dir.mkdir(parents=True, exist_ok=True)

cleaning_report_outputs = {
    "text_cleaning_report.csv": text_cleaning_report,
    "duplicate_removal_report.csv": duplicate_removal_report,
    "missing_imputation_report.csv": missing_imputation_report,
    "order_details_cleaning_report.csv": order_details_cleaning_report,
    "cleaning_after_check_report.csv": cleaning_after_check_report,
    "order_details_after_check.csv": order_details_after_check,
    "cleaning_actions_summary.csv": cleaning_actions_summary,
}

for file_name, report_df in cleaning_report_outputs.items():
    report_df.to_csv(cleaning_reports_dir / file_name, index=False, encoding="utf-8-sig")
    print(f"ذخیره شد: {cleaning_reports_dir / file_name}")


ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\text_cleaning_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\duplicate_removal_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\missing_imputation_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\order_details_cleaning_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\cleaning_after_check_report.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\order_details_after_check.csv
ذخیره شد: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\cleaning_actions_summary.csv


# تست پایانی پاکسازی

In [18]:
print("بخش پاکسازی داده‌ها و جایگزینی مقادیر گمشده با موفقیت تکمیل شد.")
print("تعداد جدول‌های موجود در tables_clean:", len(tables_clean))


بخش پاکسازی داده‌ها و جایگزینی مقادیر گمشده با موفقیت تکمیل شد.
تعداد جدول‌های موجود در tables_clean: 13
